# Batch UAV Localization (Flight 01)

End-to-end batch demo: process a UAV flight folder in sorted order, query the satellite KD-tree, and save one 3-panel figure per image.

Set `FLIGHT_ID` to switch folders. Set `TARGET_TOTAL_IMAGES` to the cumulative number of images you want processed. If you rerun later with a larger target, the notebook skips the images already handled and continues from the next one using `outputs/{FLIGHT_ID}/query_progress_{FLIGHT_ID}.json`.

The UAV images are processed from `UAV_VisLoc_dataset/{FLIGHT_ID}/drone/` in sorted filename order.

## 1. Clone repo & dependencies (skip if already done)

If you ran notebook 1 in the same session, you can jump to step 4.

In [11]:
import os

REPO_URL = 'https://github.com/kagtgi/LocalizationUAV.git'
REPO_DIR = 'LocalizationUAV'

if not os.path.exists(REPO_DIR) and not os.path.exists('localization'):
    !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}
    !pip install --quiet -r requirements.txt

## 2. Kaggle setup (only if data is not already on disk)

In [12]:
# if not os.path.exists('UAV-VisLoc/01/satellite01.tif'):
#     # Setup Kaggle API
#     !mkdir /.kaggle
#     !mv kaggle.json /.kaggle
#     !mv /.kaggle /root/
#     !chmod 600 ~/.kaggle/kaggle.json

#     !kaggle datasets download building-segment
#     !kaggle datasets download hailong1610/uav-visloc-dataset
#     !unzip -q building-segment.zip
#     !unzip -q uav-visloc-dataset.zip
#     print('UAV-VisLoc dataset downloaded and extracted')

## 2b. Download the Mask R-CNN checkpoint

`best_model.pth` (~170 MB) is hosted on Google Drive:

https://drive.google.com/file/d/1jlRfOXYU18DcOjWEwNBet1b7FHCIClcB/view

The cell below downloads it on first run and skips on subsequent runs.

In [13]:
# MODEL_GDRIVE_ID = '1jlRfOXYU18DcOjWEwNBet1b7FHCIClcB'
# MODEL_LOCAL_PATH = 'best_model.pth'

# if not os.path.exists(MODEL_LOCAL_PATH):
#     !pip install --quiet gdown
#     !gdown 'https://drive.google.com/uc?id={MODEL_GDRIVE_ID}' -O {MODEL_LOCAL_PATH}
# else:
#     print(f'{MODEL_LOCAL_PATH} already present; skipping download.')

# print('best_model.pth size:', os.path.getsize(MODEL_LOCAL_PATH) // (1024 * 1024), 'MB')

## 3. Imports + config

In [ ]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()


def _first_existing(*paths):
    for path in paths:
        if path.exists():
            return path
    raise FileNotFoundError(f'None of these paths exist: {paths}')


def _find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'localization').exists() and (candidate / 'best_model.pth').exists():
            return candidate
    raise FileNotFoundError(f'Could not find the LocalizationUAV repo above {start}')


REPO_ROOT = _find_repo_root(NOTEBOOK_DIR)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

DATA_ROOT = Path(r'D:\bk_study_stuff\EUREK_A_sao\implementation\uav_nonGPS\data\UAV_VisLoc_dataset')
FLIGHT_ID = '01'
TARGET_TOTAL_IMAGES = 30
MODEL_PATH = _first_existing(REPO_ROOT / 'best_model.pth', NOTEBOOK_DIR / 'best_model.pth')

PATCH_SIZE = 500
STRIDE = 100
SCORE_THRESHOLD = 0.5
INFERENCE_BATCH_SIZE = 4
TOP_K = 5
TOP_N_TO_SHOW = 100

OUTPUT_DIR = NOTEBOOK_DIR / 'outputs' / FLIGHT_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DB_NPZ_PATH = _first_existing(
    OUTPUT_DIR / f'satellite{FLIGHT_ID}_kdtree.npz',
    REPO_ROOT / 'outputs' / FLIGHT_ID / f'satellite{FLIGHT_ID}_kdtree.npz',
    REPO_ROOT / 'notebooks' / 'outputs' / FLIGHT_ID / f'satellite{FLIGHT_ID}_kdtree.npz',
)
DESCRIPTORS_CSV_PATH = OUTPUT_DIR / f'satellite{FLIGHT_ID}_descriptors.csv'
PROGRESS_PATH = OUTPUT_DIR / f'query_progress_{FLIGHT_ID}.json'
SUMMARY_CSV_PATH = OUTPUT_DIR / f'query_results_{FLIGHT_ID}.csv'

print('Notebook  :', NOTEBOOK_DIR)
print('Repo root :', REPO_ROOT)
print('Data root :', DATA_ROOT)
print('Flight ID :', FLIGHT_ID)
print('Target    :', TARGET_TOTAL_IMAGES)
print('DB output :', DB_NPZ_PATH)
print('State file :', PROGRESS_PATH)
assert MODEL_PATH.exists(), f'Missing Mask R-CNN checkpoint: {MODEL_PATH}'
assert DATA_ROOT.exists(), f'Missing UAV dataset root: {DATA_ROOT}'

In [ ]:
import os
import json

import numpy as np
import pandas as pd
import torch
from PIL import Image

from localization import load_model, process_uav, SatelliteDatabase, query_uav
from localization.database.builder import extract_patch_descriptors
from localization.io.bounds import (
    load_satellite_bounds,
    latlon_to_pixel,
    pixel_to_latlon,
    pixel_offset_to_meters,
)
from localization.io.dataset import VisLocFlight, load_flight_metadata, get_image_pose
from localization.matching.visualize import (
    draw_gt_and_prediction,
    draw_gt_vs_topn_centroids,
    render_top_n_result,
)

flight = VisLocFlight(flight_id=FLIGHT_ID, root=DATA_ROOT)
print('Satellite TIF :', flight.satellite_tif)
print('Metadata CSV  :', flight.metadata_csv)
print('Bounds CSV    :', flight.bounds_csv)
assert flight.satellite_tif.exists(), f'Missing satellite TIF: {flight.satellite_tif}'
assert flight.drone_dir.exists(), f'Missing drone folder: {flight.drone_dir}'


## 4. Load Mask R-CNN + satellite database

In [16]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = load_model(
    model_path=str(MODEL_PATH),
    device=device,
    num_classes=2,
    pretrained=False,
).to(device).eval()
print('Device:', device)

db = SatelliteDatabase.load(str(DB_NPZ_PATH))
print(db)

Device: cuda
SatelliteDatabase(size=17587600, patches=22477, parent_tif='satellite01.tif', leaf_size=40)


## 5. Batch query loop

Process the drone folder in sorted order, resume from `query_progress_{FLIGHT_ID}.json`, and stop after `TARGET_TOTAL_IMAGES` cumulative images have been handled.

In [ ]:
def _load_progress(path: Path) -> dict:
    if path.exists():
        return json.loads(path.read_text(encoding='utf-8'))
    return {'processed_count': 0, 'processed_images': []}


def _save_progress(path: Path, state: dict) -> None:
    path.write_text(json.dumps(state, indent=2, ensure_ascii=False), encoding='utf-8')


def _list_drone_images(flight: VisLocFlight) -> list[Path]:
    return sorted(
        p for p in flight.drone_dir.iterdir()
        if p.is_file() and p.suffix.lower() in {'.jpg', '.jpeg', '.png'}
    )


def _select_next_batch(images: list[Path], state: dict, target_total: int) -> list[Path]:
    done = min(int(state.get('processed_count', 0)), len(images))
    target_total = min(int(target_total), len(images))
    if target_total <= done:
        return []
    return images[done:target_total]


drone_images = _list_drone_images(flight)
progress = _load_progress(PROGRESS_PATH)
samples = _select_next_batch(drone_images, progress, TARGET_TOTAL_IMAGES)
print(f'{len(drone_images)} drone images available in flight {FLIGHT_ID}')
print('Already processed:', int(progress.get('processed_count', 0)))
print('This run will process:', len(samples))
for s in samples[:5]:
    print('  next:', s.name)
if len(samples) > 5:
    print('  ...')
assert len(samples) > 0, 'No new images to process for the requested TARGET_TOTAL_IMAGES.'

bounds = load_satellite_bounds(
    satellite_filename=os.path.basename(str(flight.satellite_tif)),
    csv_path=str(flight.bounds_csv),
)

with Image.open(flight.satellite_tif) as sat:
    sat_w, sat_h = sat.size
print(f'Satellite size: {sat_w} x {sat_h}')
print('Bounds:', bounds)
metadata_df = load_flight_metadata(flight.metadata_csv)
print('Metadata rows :', len(metadata_df))

results_summary = []
start_index = int(progress.get('processed_count', 0))

for idx, sample_path in enumerate(samples, start=start_index + 1):
    image_name = sample_path.name
    print('=' * 70)
    print(f'Processing {idx}/{min(TARGET_TOTAL_IMAGES, len(drone_images))}:', image_name)

    try:
        _, img_uav_500, meta = process_uav(
            img_path=str(sample_path),
            csv_path=str(flight.metadata_csv),
        )
        print(f'  Pose: yaw={meta.get("Yaw (Phi)", float("nan")):.1f}, height={meta.get("height", float("nan")):.1f}')

        uav_descriptors, _ = extract_patch_descriptors(
            patch_image=img_uav_500,
            model=model,
            device=device,
            score_threshold=SCORE_THRESHOLD,
        )
        if uav_descriptors.shape[0] == 0:
            print('  No buildings segmented; skipping this sample.')
            results_summary.append({
                'index': idx,
                'image': image_name,
                'descriptors': 0,
                'rank1_patch': None,
                'rank1_votes': None,
                'margin': None,
                'error_m': None,
                'status': 'no_descriptors',
            })
            progress['processed_count'] = idx
            progress.setdefault('processed_images', []).append(image_name)
            _save_progress(PROGRESS_PATH, progress)
            continue

        print(f'  UAV descriptors: {uav_descriptors.shape[0]}')
        result = query_uav(uav_descriptors, db, k=TOP_K, top_n=TOP_N_TO_SHOW)
        if result is None:
            print('  Query produced no winner; skipping.')
            results_summary.append({
                'index': idx,
                'image': image_name,
                'descriptors': int(uav_descriptors.shape[0]),
                'rank1_patch': None,
                'rank1_votes': None,
                'margin': None,
                'error_m': None,
                'status': 'no_winner',
            })
            progress['processed_count'] = idx
            progress.setdefault('processed_images', []).append(image_name)
            _save_progress(PROGRESS_PATH, progress)
            continue

        print(f'  Rank-1 patch  : {result.patch_id}  (votes={result.vote_count}, margin={result.margin})')

        gt_pixel = None
        error_distance_m = None
        row = get_image_pose(metadata_df, image_name)
        if row is not None and bounds is not None:
            gt_lat, gt_lon = float(row['lat']), float(row['lon'])
            gt_pixel = latlon_to_pixel(gt_lat, gt_lon, bounds, sat_w, sat_h)
            offset = pixel_offset_to_meters(
                result.pixel_xy[0] - gt_pixel[0],
                result.pixel_xy[1] - gt_pixel[1],
                bounds, sat_w, sat_h,
            )
            error_distance_m = float(offset['distance_m'])
            print(f'  GT pixel      : {gt_pixel}')
            print(f'  Error vs GT   : {error_distance_m:.1f} m (dx={offset["dx_m"]:.1f}, dy={offset["dy_m"]:.1f})')

        out_path = OUTPUT_DIR / f'match_top{TOP_N_TO_SHOW}_{sample_path.stem}.png'
        title = f'Top {TOP_N_TO_SHOW} candidates: {image_name}  ->  rank-1 = {result.patch_id}'
        if error_distance_m is not None:
            title += f'   (error: {error_distance_m:.1f} m)'

        fig = render_top_n_result(
            uav_image_path=str(sample_path),
            satellite_image_path=str(flight.satellite_tif),
            top_n=result.top_n,
            gt_pixel_xy=gt_pixel,
            error_distance_m=error_distance_m,
            zoom_radius_px=1500,
            highlight_top_k=5,
            title=title,
            output_path=str(out_path),
        )
        print('  Figure saved to:', out_path)

        gt_vs_top_path = OUTPUT_DIR / f'gt_vs_top{TOP_N_TO_SHOW}_{sample_path.stem}.png'
        height = meta.get('height')
        phi = meta.get('Yaw (Phi)')
        fig5a_title = f'Satellite map ({FLIGHT_ID}'
        if height is not None:
            fig5a_title += f', {height:.0f} m'
        if phi is not None:
            fig5a_title += f', phi={phi:.0f} deg'
        fig5a_title += ')'
        draw_gt_vs_topn_centroids(
            satellite_image_path=str(flight.satellite_tif),
            top_n=result.top_n,
            gt_pixel_xy=gt_pixel,
            title=fig5a_title,
            output_path=str(gt_vs_top_path),
        )
        print('  GT-vs-top-N map saved to:', gt_vs_top_path)

        overlay_path = OUTPUT_DIR / f'overlay_{sample_path.stem}.jpg'
        sat_overlay = draw_gt_and_prediction(
            satellite_image_path=str(flight.satellite_tif),
            predicted_pixel_xy=result.pixel_xy,
            gt_pixel_xy=gt_pixel,
            error_distance_m=error_distance_m,
        )
        import cv2
        cv2.imwrite(str(overlay_path), cv2.cvtColor(sat_overlay, cv2.COLOR_RGB2BGR))
        print('  Satellite overlay saved to:', overlay_path)

        results_summary.append({
            'index': idx,
            'image': image_name,
            'descriptors': int(uav_descriptors.shape[0]),
            'rank1_patch': result.patch_id,
            'rank1_votes': int(result.vote_count),
            'margin': int(result.margin),
            'error_m': error_distance_m,
            'status': 'ok',
        })

        progress['processed_count'] = idx
        progress.setdefault('processed_images', []).append(image_name)
        _save_progress(PROGRESS_PATH, progress)

    except Exception as exc:
        print('  Failed:', exc)
        results_summary.append({
            'index': idx,
            'image': image_name,
            'descriptors': None,
            'rank1_patch': None,
            'rank1_votes': None,
            'margin': None,
            'error_m': None,
            'status': f'error: {exc}',
        })
        progress['processed_count'] = idx
        progress.setdefault('processed_images', []).append(image_name)
        _save_progress(PROGRESS_PATH, progress)
        continue

summary_df = pd.DataFrame(results_summary)
summary_df.to_csv(SUMMARY_CSV_PATH, index=False)
print('=' * 70)
print('All requested samples processed.')
print('Summary saved to:', SUMMARY_CSV_PATH)
summary_df